# File Operations: Safe, Atomic, and Secure

This notebook demonstrates `siege_utilities.files` — the file operations
package providing atomic writes, security validation, and safe command execution.

## What this covers
1. Atomic file writes (crash-safe)
2. Path security validation (traversal prevention)
3. Safe JSON/text read-write operations
4. Secure command execution with allow-lists

## 1. Atomic Writes

The `atomic_write_path` context manager writes to a temp file, then atomically
renames to the target. If anything fails, the original is untouched.

In [1]:
from pathlib import Path
import tempfile
from siege_utilities.files.operations import atomic_write_path

target = Path(tempfile.mkdtemp()) / "config.json"
target.write_text('{"version": 1}')
print(f"Before: {target.read_text()}")

# Atomic write — if this succeeds, target is updated
with atomic_write_path(target) as tmp:
    tmp.write_text('{"version": 2}')
print(f"After:  {target.read_text()}")

# If the block raises, original is preserved
try:
    with atomic_write_path(target) as tmp:
        tmp.write_text('{"version": 3, "broken": ')
        raise RuntimeError("simulated crash")
except RuntimeError:
    pass
print(f"After crash: {target.read_text()}  # still version 2")

Before: {"version": 1}
After:  {"version": 2}
After crash: {"version": 2}  # still version 2


## 2. Path Security Validation

The `validation` module blocks path traversal, null byte injection,
and access to sensitive system files.

In [2]:
from siege_utilities.files.validation import (
    validate_safe_path,
    is_path_traversal_attempt,
    is_sensitive_path,
    PathSecurityError,
)

# Traversal detection
print(f"'../etc/passwd' is traversal: {is_path_traversal_attempt('../etc/passwd')}")
print(f"'data/file.txt' is traversal: {is_path_traversal_attempt('data/file.txt')}")

# Sensitive path detection
print(f"'/etc/passwd' is sensitive: {is_sensitive_path('/etc/passwd')}")
print(f"'/tmp/data.csv' is sensitive: {is_sensitive_path('/tmp/data.csv')}")

# Validation blocks dangerous paths
try:
    validate_safe_path('../../etc/shadow')
except PathSecurityError as e:
    print(f"Blocked: {e}")

'../etc/passwd' is traversal: True
'data/file.txt' is traversal: False
'/etc/passwd' is sensitive: True
'/tmp/data.csv' is sensitive: False
Blocked: Path traversal attempt blocked: ../../etc/shadow


## 3. Safe JSON/Text Operations

These functions handle encoding, directory creation, and error
recovery automatically.

In [3]:
from siege_utilities.files.operations import (
    safe_json_write, safe_json_read,
    safe_file_write, safe_file_read,
    ensure_directory_exists,
)

tmp = Path(tempfile.mkdtemp())

# JSON round-trip (creates parent dirs automatically)
config = {'project': 'Demographics', 'state_fips': '48', 'vintage': 2020}
safe_json_write(str(tmp / 'deep' / 'nested' / 'config.json'), config)
loaded = safe_json_read(str(tmp / 'deep' / 'nested' / 'config.json'))
print(f"Round-trip: {loaded}")

# Missing files return None (not empty dict)
result = safe_json_read(str(tmp / 'nonexistent.json'))
print(f"Missing file: {result}  # None, not {{}}")

Round-trip: {'project': 'Demographics', 'state_fips': '48', 'vintage': 2020}
Missing file: None  # None, not {}


## 4. Secure Command Execution

`run_command` validates commands against an allow-list before execution.
Shell metacharacters are blocked. No shell=True.

In [4]:
from siege_utilities.files.operations import run_command
from siege_utilities.files.shell import SecurityError

# Allowed command succeeds
result = run_command('echo Hello from siege_utilities')
print(f"stdout: {result.stdout.strip()}")

# Disallowed commands are blocked
try:
    run_command('python -c "import os; os.system(\"rm -rf /\")"')
except SecurityError as e:
    print(f"Blocked: {e}")

# Custom allow-list for specific workflows
result = run_command('git status', allow_list={'git', 'ls'})
print(f"git status exit code: {result.returncode}")

Security validation failed: Command 'python' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']


stdout: Hello from siege_utilities
Blocked: Command 'python' not allowed. Allowed commands: ['cat', 'date', 'echo', 'find', 'grep', 'head', 'hostname', 'ls', 'pwd', 'tail', 'uname', 'wc', 'which', 'whoami']
git status exit code: 0


## Summary

The `files/` package provides the safety infrastructure that other modules
build on:

- **Atomic writes** prevent corruption on crashes
- **Path validation** blocks traversal and injection attacks
- **Safe I/O** handles encoding, dirs, and errors
- **Secure execution** blocks command injection by default